# Stage 3 G0 controls (CPU)
Run preparation once first, then consume its explicit plan and shared assignment
reference and each ordinary worker's statistics references. Use disjoint group/seed
ownership across runtimes. This notebook never creates statistics/assignments,
scores ordinary systems or opens CONFIRM. It does not validate Stage 3 completion.

Start with the prescribed representative seed0 C=.01 blocks and reuse their indexes
for full fitting after reviewing costs alongside ordinary C=.01/C=10 measurements.
The full mode always uses all four Cs; seeds are only a distributed ownership split.
After all three seed scores exist, the shared engine uses 100-draw RAM batches and
publishes one all-1999-draw summary per group. Interrupted unfinished null work may
restart; completed C blocks, scores and null groups are reused from explicit refs.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 and the frozen CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
from google.colab import drive
drive.mount('/content/drive')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO = Path('/content/Research-stage3-g0')
CODE_REF = 'phase1'  # Resolve once below; replace with a published worker SHA to pin explicitly.
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local edits before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO/'requirements-stage2-cpu.txt')], check=True)
print('Fixed worker commit:', CODE_COMMIT)
print('If installation requests a restart, restart and rerun setup.')


In [ ]:
# Copy immutable references printed by preparation/ordinary/G0 workers. No latest-file discovery.
PLAN_MANIFEST = None  # PHASE1_ROOT / 'cache/shared/stage3-plans/<run>/_COMPLETE.json'
ASSIGNMENTS = None  # {'path': '...', 'sha256': '...'} from preparation
FEATURE_SOURCE = 'vlm'  # vlm or e03; E03 runs once across both models
MODEL = 'qwen'  # qwen/gemma for vlm; set None for e03
GROUP_IDS = []  # exact group IDs from the plan; explicit disjoint ownership
SEEDS = [0]  # full default ownership after profiling: [0, 1, 2]
PROFILE = True  # seed0 C=.01 on metadata representatives only; full mode uses all four Cs
REDUCE_ONLY = False  # True with GROUP_IDS=[], SEEDS=[], PROFILE=False
STATISTICS_REFS = []  # ordinary-owned group statistics
STATISTICS_INDEXES = []  # ordinary indexes; only their statistics entries are consumed
RESUME_INDEXES = []  # G0 group/failure indexes, including representative C blocks


In [ ]:
if PLAN_MANIFEST is None or ASSIGNMENTS is None:
    raise RuntimeError('Set preparation-owned plan and assignments references.')
if FEATURE_SOURCE not in ('vlm', 'e03') or (FEATURE_SOURCE == 'e03' and MODEL is not None) or (FEATURE_SOURCE == 'vlm' and MODEL not in ('qwen', 'gemma')):
    raise RuntimeError('Set FEATURE_SOURCE and MODEL consistently; E03 has no model selector.')
if not GROUP_IDS and not REDUCE_ONLY:
    raise RuntimeError('Declare exact disjoint group ownership, or explicitly choose reduction only.')
if REDUCE_ONLY and (GROUP_IDS or SEEDS or PROFILE):
    raise RuntimeError('Reduction only requires empty group/seed ownership and PROFILE=False.')
if any(type(seed) is not int or seed not in (0, 1, 2) for seed in SEEDS):
    raise RuntimeError('Only fixed seeds 0,1,2 are supported.')
if subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip() != CODE_COMMIT or subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip():
    raise RuntimeError('Checkout changed after setup.')
# Inspect in an isolated child with the same numerical import path as the worker;
# notebook imports must not affect its thread/runtime identity.
inspect_code = """import json,sys
from pathlib import Path
from src.stage3.interfaces import load_fit_plan
d=json.load(sys.stdin)
protocol=json.loads(Path('configs/protocol.json').read_text(encoding='utf-8'))
p=load_fit_plan(d['manifest'],d['root'],protocol)
for gid in d['groups']:
    g=p['groups'][gid]
    if g['source']!=d['source'] or (d['source']=='vlm' and g['model']!=d['model']):
        raise RuntimeError('Group selector/ownership mismatch: '+gid)
    if d['profile'] and gid not in p['profiling_groups']:
        raise RuntimeError('Profile requires preparation-selected representatives')
print(json.dumps({'profiling_groups':list(p['profiling_groups']),'owned_groups':d['groups']}))
"""
subprocess.run([sys.executable, '-c', inspect_code], cwd=REPO,
               input=json.dumps({'manifest':str(PLAN_MANIFEST),'root':str(PHASE1_ROOT),'groups':GROUP_IDS,
                                 'source':FEATURE_SOURCE,'model':MODEL,'profile':PROFILE}),
               text=True, check=True, env=os.environ.copy())
settings = {'groups':GROUP_IDS,'seeds':['seed'+str(s) for s in SEEDS], 'assignments':ASSIGNMENTS,
            'supplied':{'statistics':STATISTICS_REFS},'statistics_indexes':STATISTICS_INDEXES,
            'resume_indexes':RESUME_INDEXES,'profile':PROFILE}
# Local configuration only, outside the clean source checkout. Immutable scientific
# checkpoints and recovery indexes are published by the worker under PHASE1_ROOT.
import tempfile
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', encoding='utf-8', delete=False) as handle:
    json.dump(settings, handle)
    SETTINGS_PATH = Path(handle.name)
command = [sys.executable, '-u', '-m', 'src.stage3.run_g0', '--phase1-root', str(PHASE1_ROOT),
           '--manifest', str(PLAN_MANIFEST), '--settings', str(SETTINGS_PATH), '--storage-scope', 'persistent-drive']
process = subprocess.Popen(command, cwd=REPO, env=os.environ.copy(), stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    status = process.wait()
except KeyboardInterrupt:
    import signal
    process.send_signal(signal.SIGINT)
    process.wait()
    raise
finally:
    SETTINGS_PATH.unlink(missing_ok=True)
if status:
    raise RuntimeError({2:'Blocked/failed; use streamed failure index.',3:'Some groups await ordinary statistics; ready groups were retained.',4:'Global diagnostic flagged; preserve evidence and investigate.'}.get(status,'Worker failed.'))
print('Worker finished its declared ownership. Review coverage/gate and retain printed indexes; Stage 3 acceptance remains separate.')


Retain each `group_checkpoint` / `failure_checkpoint` index and pass it in
`RESUME_INDEXES` on restart. Exit 0 may represent partial group/seed coverage;
`coverage_complete` and the explicit gate reference identify whole-family coverage.
A flagged diagnostic returns exit 4 without deleting artifacts or rerolling seeds.
Metrics and nulls are control diagnostics only. Mounted production timings must be
measured here; synthetic fixture costs do not predict full-stage runtime.
